# Ch 32 부록 — diffusion LM 붕괴의 진짜 범인을 직접 찾기: 데이터로더

> 본 챕터(Ch 32)에서 작은 diffusion LM이 영어 동화를 생성하는 걸 봤습니다. 그런데 그 레시피(vocab 2048 · 30000 step · `1/t` loss)에 도달하기까지, **vocab도 step도 다 맞는데 모델이 계속 유니그램으로 붕괴하던** 시기가 있었습니다. "vocab이 크다", "학습이 부족하다" 같은 그럴듯한 가설을 다 고쳤는데도요. 진짜 범인은 뜻밖에도 **데이터로더 설정** 이었습니다.

이 부록은 그 붕괴를 **직접 재현하고 → 눈으로 진단하고 → 한 줄로 고칩니다.**

**이 부록에서 직접 경험할 것**
1. 💥 **붕괴 재현** — `num_workers=2` + generator 없는 콜레이터로 학습 → loss가 유니그램 값에서 평탄
2. 🔬 **진단** — DataLoader가 내놓는 마스킹 패턴을 직접 찍어 보면, 워커끼리 *같은 패턴을 반복* 함
3. 🛠️ **수정** — 콜레이터에 generator를 주고 `num_workers=0` → loss 하락, 생성 회복

**환경**: Google Colab **T4 GPU**. **예상 소요**: 약 18분 (데이터 + 학습 2회를 경량 step으로).

> ⚠️ *경량 재현* 입니다. 완전한 coherent 생성은 30000 step(본 챕터)이 필요하고, 이 부록은 짧은 step으로 *붕괴 vs 정상* 의 **방향**(loss·복원 정확도)만 비교합니다.

## 0. 환경 셋업

In [ ]:
%pip install -q -U transformers tokenizers datasets accelerate

import math, time, warnings, torch
import torch.nn.functional as F
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
STEPS = 8000          # 경량 — 붕괴 vs 정상의 방향만 보면 됨 (본 챕터는 30000)
print("device:", device, "| fp16:", USE_FP16, "| steps:", STEPS)

## 1. 데이터 + BPE 2048 — 본 챕터와 동일 (가설은 이미 다 맞음)

vocab은 2048(작게), step도 충분히 둘 겁니다. 즉 *흔한 가설은 다 통제* 한 상태에서 출발합니다.

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

raw_train = load_dataset("roneneldan/TinyStories", split="train[:30000]")
raw_val   = load_dataset("roneneldan/TinyStories", split="validation[:500]")

VOCAB = 2048
def corpus_iter(bs=1000):
    for i in range(0, len(raw_train), bs):
        yield raw_train[i:i+bs]["text"]
_tk = Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
_tk.decoder = decoders.ByteLevel()
_tk.train_from_iterator(corpus_iter(), trainer=trainers.BpeTrainer(
    vocab_size=VOCAB, special_tokens=["[PAD]", "[UNK]", "[MASK]"]))
tokenizer = PreTrainedTokenizerFast(tokenizer_object=_tk, pad_token="[PAD]",
                                    unk_token="[UNK]", mask_token="[MASK]")

BLOCK = 128
def tok_fn(b): return {"input_ids": tokenizer(b["text"], add_special_tokens=False)["input_ids"]}
def group(b):
    cat = sum(b["input_ids"], []); n = (len(cat)//BLOCK)*BLOCK
    return {"input_ids": [cat[i:i+BLOCK] for i in range(0, n, BLOCK)]}
def prep(ds):
    d = ds.map(tok_fn, batched=True, remove_columns=ds.column_names)
    return d.map(group, batched=True)
lm_train = prep(raw_train); lm_val = prep(raw_val)
lm_train = lm_train.remove_columns([c for c in lm_train.column_names if c != "input_ids"])
lm_val   = lm_val.remove_columns([c for c in lm_val.column_names if c != "input_ids"])
print(f"vocab {tokenizer.vocab_size} | train chunks {len(lm_train):,} | baseline ln(V)={math.log(VOCAB):.2f}")

## 2. 모델·loss·평가 함수 (본 챕터와 동일, 공통)

작은 `BertForMaskedLM`(hidden 256/4L) + `1/t` 시간가중 loss. 붕괴 실험과 정상 실험이 *똑같은* 모델·loss 를 쓰도록 함수로 묶습니다.

In [ ]:
from transformers import BertConfig, BertForMaskedLM, Trainer, TrainingArguments

def new_model():
    cfg = BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
                     num_attention_heads=4, intermediate_size=1024,
                     max_position_embeddings=BLOCK, pad_token_id=tokenizer.pad_token_id)
    return BertForMaskedLM(cfg).to(device)

class DiffusionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        t = inputs["t"]; labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        B, L, V = out.logits.shape
        per = F.cross_entropy(out.logits.view(-1, V), labels.view(-1),
                              ignore_index=-100, reduction="none").view(B, L)
        loss = ((per.sum(1)/L) / t.to(per.dtype)).mean()
        return (loss, out) if return_outputs else loss

def train_args(tag, num_workers):
    return TrainingArguments(output_dir=f"./out_{tag}", max_steps=STEPS,
        per_device_train_batch_size=64, learning_rate=3e-4, weight_decay=0.01,
        warmup_steps=500, lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
        logging_steps=200, save_strategy="no", report_to="none", label_names=["labels"],
        remove_unused_columns=False, dataloader_num_workers=num_workers, seed=SEED)

g_eval = torch.Generator().manual_seed(0)
@torch.no_grad()
def fixed_t_acc(model, tv=0.15, n=128):
    # 고정 비율(15%) 마스킹 복원 top-1 정확도 — 샘플러 무관 모델 품질 지표
    model.eval(); cor = tot = 0
    for ex in lm_val.select(range(min(n, len(lm_val)))):
        ids = torch.tensor(ex["input_ids"]); m = torch.rand(len(ids), generator=g_eval) < tv
        if not m.any(): m[0] = True
        inp = ids.clone(); inp[m] = tokenizer.mask_token_id
        pr = model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor += (pr[m] == ids[m]).sum().item(); tot += int(m.sum())
    return cor / tot
print("준비 완료")

## 3. 💥 시도 1 — 붕괴 재현: `num_workers=2` + generator 없는 콜레이터

가장 자연스러워 보이는 콜레이터입니다. 매 배치 `torch.rand` 로 마스킹 비율 `t` 와 마스킹 위치를 뽑습니다 — *generator 인자 없이, 전역 RNG* 로요. 그리고 데이터 로딩을 빠르게 하려고 `dataloader_num_workers=2` 를 켭니다. 흔히 하는 설정이죠.

vocab 도 2048(작게), step 도 8000(충분히) 입니다. *흔한 가설은 다 통제* 했습니다. 그런데 학습해 보면…

In [ ]:
class NaiveCollator:
    '''generator 없이 전역 torch.rand 로 마스킹 — num_workers>0 에서 문제가 된다.'''
    def __init__(self, tokenizer, eps=0.02):
        self.mask_id = tokenizer.mask_token_id; self.eps = eps
    def __call__(self, ex):
        ids = torch.tensor([e["input_ids"] for e in ex], dtype=torch.long); B, L = ids.shape
        t = torch.rand(B) * (1.0 - self.eps) + self.eps              # 전역 RNG (generator 없음)
        mask = torch.rand(B, L) < t.unsqueeze(1)                     # 전역 RNG
        no = ~mask.any(1)
        if no.any(): mask[no, torch.randint(0, L, (int(no.sum()),))] = True
        inp = ids.clone(); inp[mask] = self.mask_id
        lab = ids.clone(); lab[~mask] = -100
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long),
                "labels": lab, "t": t}

torch.manual_seed(SEED)
model_bad = new_model()
out_bad = DiffusionTrainer(model=model_bad, args=train_args("bad", num_workers=2),
                           train_dataset=lm_train, data_collator=NaiveCollator(tokenizer)).train()
acc_bad = fixed_t_acc(model_bad)
print(f"\n[붕괴 시도] train_loss {out_bad.training_loss:.3f}  (baseline ln(V)={math.log(VOCAB):.2f})")
print(f"           fixed-t(0.15) top-1 acc {acc_bad:.3f}")
print("→ loss 가 유니그램 값 근처에서 거의 안 내려가고, 복원 정확도도 낮습니다 (유니그램 붕괴).")

## 4. 🔬 진단 — 왜? DataLoader가 내놓는 마스킹을 직접 찍어 보자

vocab·step·loss 가 다 맞는데 왜 붕괴할까요? 범인은 **`num_workers=2` + generator 없는 콜레이터** 의 조합입니다. PyTorch DataLoader는 `num_workers>0` 이면 워커 프로세스를 **fork** 하는데, 콜레이터가 전역 RNG를 쓰면 워커들이 *같은 난수 상태를 물려받아* 마스킹 패턴이 상관됩니다. 그러면 매 배치 *비슷한 자리* 만 가려져, 모델이 다양한 빈칸 채우기를 못 배우고 유니그램으로 주저앉습니다.

직접 확인합니다 — `num_workers=2` 로 몇 배치를 뽑아 *마스킹 비율 t* 를 찍어 보면, 워커 주기로 값이 **반복** 됩니다.

In [ ]:
from torch.utils.data import DataLoader

def first_t_values(collator, num_workers, n_batches=6):
    dl = DataLoader(lm_train, batch_size=4, collate_fn=collator,
                    num_workers=num_workers, shuffle=False)
    out = []
    for i, b in enumerate(dl):
        out.append([round(float(x), 3) for x in b["t"]])
        if i + 1 >= n_batches: break
    return out

print("=== NaiveCollator + num_workers=2 (붕괴 설정) — 배치별 마스킹 비율 t ===")
for i, ts in enumerate(first_t_values(NaiveCollator(tokenizer), num_workers=2)):
    print(f"  batch {i}: {ts}")
print("  ↑ 워커가 fork 된 RNG 를 공유해 같은 t 패턴이 주기적으로 반복됩니다 (다양성 붕괴).")

## 5. 🛠️ 수정 — 콜레이터에 generator를 주고 `num_workers=0`

해결은 두 가지로, 둘 다 *마스킹 난수를 워커와 독립* 시킵니다.
- 콜레이터가 **자기 `torch.Generator`** 를 들고 거기서 난수를 뽑습니다 (전역 RNG 미사용).
- `dataloader_num_workers=0` 으로 메인 프로세스에서 콜레이트합니다 (fork 자체를 없앰).

먼저 같은 진단을 새 콜레이터로 돌려, t 가 *반복 없이 다양* 한지 확인합니다.

In [ ]:
class DiffusionCollator:
    '''자기 generator 로 마스킹 — 워커와 독립, 매 배치 다양.'''
    def __init__(self, tokenizer, eps=0.02, seed=42):
        self.mask_id = tokenizer.mask_token_id; self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)
    def __call__(self, ex):
        ids = torch.tensor([e["input_ids"] for e in ex], dtype=torch.long); B, L = ids.shape
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps
        mask = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)
        no = ~mask.any(1)
        if no.any(): mask[no, torch.randint(0, L, (int(no.sum()),), generator=self.gen)] = True
        inp = ids.clone(); inp[mask] = self.mask_id
        lab = ids.clone(); lab[~mask] = -100
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long),
                "labels": lab, "t": t}

print("=== DiffusionCollator + num_workers=0 (정상 설정) — 배치별 마스킹 비율 t ===")
for i, ts in enumerate(first_t_values(DiffusionCollator(tokenizer), num_workers=0)):
    print(f"  batch {i}: {ts}")
print("  ↑ 반복 없이 매 배치 다른 t — 모델이 다양한 마스킹을 보고 문맥을 배웁니다.")

## 6. ✅ 정상 학습 — 같은 모델·step, 콜레이터·num_workers만 고침

In [ ]:
torch.manual_seed(SEED)
model_good = new_model()
out_good = DiffusionTrainer(model=model_good, args=train_args("good", num_workers=0),
                            train_dataset=lm_train, data_collator=DiffusionCollator(tokenizer)).train()
acc_good = fixed_t_acc(model_good)
print(f"\n=== 붕괴 vs 정상 (vocab·step·loss 동일, 데이터로더만 차이) ===")
print(f"붕괴(naive + workers=2) : loss {out_bad.training_loss:.3f}   acc {acc_bad:.3f}")
print(f"정상(gen   + workers=0) : loss {out_good.training_loss:.3f}   acc {acc_good:.3f}")

In [ ]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.bar(["collapse\n(naive, nw=2)", "fixed\n(gen, nw=0)"], [out_bad.training_loss, out_good.training_loss],
       color=["tab:red", "tab:green"], alpha=0.85)
a1.axhline(math.log(VOCAB), ls="--", color="gray", label=f"ln(V)={math.log(VOCAB):.2f}")
a1.set_title("train loss (lower = better)"); a1.legend()
a2.bar(["collapse\n(naive, nw=2)", "fixed\n(gen, nw=0)"], [acc_bad, acc_good],
       color=["tab:red", "tab:green"], alpha=0.85)
a2.set_title("fixed-t(0.15) top-1 accuracy (higher = better)")
plt.tight_layout(); plt.show()

## 7. 정리 — 무엇을 배웠나

직접 겪은 순서:

1. **붕괴 재현**: vocab 2048·step 8000 으로 *흔한 가설을 다 통제* 했는데도, `num_workers=2` + generator 없는 콜레이터로 학습하니 loss 가 유니그램 값에서 평탄 (붕괴).
2. **진단**: DataLoader가 내놓는 `t` 를 직접 찍어 보니, 워커가 fork 된 RNG 를 공유해 마스킹 패턴이 *반복* 됨 → 모델이 다양한 빈칸 채우기를 못 배움.
3. **수정**: 콜레이터에 자기 generator + `num_workers=0` → 마스킹이 다양해지고 loss 하락·복원 정확도 상승.

**핵심 교훈**
- 학습이 무너질 때, 범인이 늘 *모델·vocab·step* 같은 "큰 것" 에 있는 건 아닙니다. **데이터 파이프라인(난수·셔플·콜레이트)** 이 조용히 망가뜨릴 수 있습니다.
- diffusion(과 MLM) 의 학습 신호는 **마스킹의 *다양성*** 입니다. 매 배치 *어디를 가리는지* 가 충분히 달라야 모델이 문맥을 배웁니다 — 같은 패턴이 반복되면 유니그램으로 주저앉습니다.
- 무작위성을 쓰는 콜레이터는 **명시적 `torch.Generator`** 로 워커와 독립시키세요. 이 한 줄이 본 챕터의 정상 학습과 이 부록의 붕괴를 가릅니다.